# 🔑 INSTRUCTOR ANSWER KEY — do not distribute

# 📏 Microproject 2: Galton's Heights
*Elements of Data Science — Regression Microprojects* <br>

In the 1880s Francis Galton measured the heights of 934 adult children and their parents. Studying these data, he noticed that children of very tall parents were tall — but usually *less* extreme than their parents. He called this "regression toward mediocrity," which is where the word **regression** comes from.

> **Your research question:** How well do parents' heights predict a child's adult height — and why do predictions 'regress' toward the average?

In Lab 8 you built a **regression toolbox** step by step. This microproject gives you much less scaffolding: you decide how to explore, you build the line, and you judge whether it is any good. Your final product is **one publication-quality figure**, submitted to the class gallery, that your instructor will show in class.

**Review:** [Ch. 15.2 The Regression Line](https://inferentialthinking.com/chapters/15/2/Regression_Line.html) (uses these same data), [Ch. 15.1 Correlation](https://inferentialthinking.com/chapters/15/1/Correlation.html)

|Part|What you do|
|---|---|
|1|Rebuild your regression toolbox|
|2|Meet the data|
|3|Look first: scatter plot and a guess|
|4|Fit the line and interpret it|
|5|Check the fit: errors and `np.polyfit`|
|6|The twist: dig deeper|
|7|Make and submit your final figure|

## ⚙️ Initialization — DO NOT EDIT
👉 **Run the cell below first** (Shift+Enter) before anything else.

In [ ]:
# 🔧 Run this cell first! (Shift+Enter)
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "shared-public")))
sys.path.insert(0, "/home/jovyan/shared-public")
from init import *
from microproject_tools import my_project, submit_plot

### 🧰 Standard imports

In [ ]:
import numpy as np
from datascience import *
%matplotlib inline
import matplotlib.pyplot as plt

Enter your name (first and last), then check that this is the project you were assigned.

In [ ]:
name = 'Test Student'
my_project(name)

## 1. Your Regression Toolbox
Copy your four functions from **Lab 8, Questions 1.5 and 4.4** into the cell below (or rebuild them from [Ch. 15.2](https://inferentialthinking.com/chapters/15/2/Regression_Line.html)). Then run the test cell: a perfect line $y = 2x + 1$ must give $r = 1$, slope 2, intercept 1.

In [ ]:
def standard_units(any_numbers):
    "Convert an array of numbers to standard units."
    return (any_numbers - np.mean(any_numbers)) / np.std(any_numbers)

def correlation(t, label_x, label_y):
    "The correlation coefficient r between two columns of Table t."
    return np.mean(standard_units(t.column(label_x)) * standard_units(t.column(label_y)))

def slope(t, label_x, label_y):
    "Slope of the regression line, in original units."
    r = correlation(t, label_x, label_y)
    return r * np.std(t.column(label_y)) / np.std(t.column(label_x))

def intercept(t, label_x, label_y):
    "Intercept of the regression line, in original units."
    return np.mean(t.column(label_y)) - slope(t, label_x, label_y) * np.mean(t.column(label_x))

In [ ]:
# Toolbox test — don't change this cell
test = Table().with_columns('x', make_array(1, 2, 3, 4), 'y', make_array(3, 5, 7, 9))
print('r =', correlation(test, 'x', 'y'), '  slope =', slope(test, 'x', 'y'), '  intercept =', intercept(test, 'x', 'y'))
print('✅ Toolbox works!' if np.isclose(correlation(test, 'x', 'y'), 1) and np.isclose(slope(test, 'x', 'y'), 2)
      and np.isclose(intercept(test, 'x', 'y'), 1) else '❌ Check your functions against Lab 8.')

## 2. Meet the Data
One row per adult child, heights in **inches**:
- **family**: family ID
- **father**, **mother**
- **midparent**: a weighted average of the parents' heights (Galton multiplied mothers' heights by 1.08 to compare with fathers)
- **gender** of the child (`male` / `female`)
- **child**: the child's adult height

*Source:* Galton (1886), as distributed with the *Inferential Thinking* textbook ([data-8/textbook](https://github.com/data-8/textbook)).

In [ ]:
data = Table.read_table('galton_heights.csv')
data

<font color=blue>**Explore.**</font> How many families are there? How many sons and how many daughters? (Try `group`.) Use any Table methods you know (`num_rows`, `sort`, `where`, `group`, `hist`...). Add more cells if you like.

In [ ]:
print(data.num_rows, 'rows')
print('x range:', min(data.column('midparent')), 'to', max(data.column('midparent')))
print('y range:', min(data.column('child')), 'to', max(data.column('child')))

## 3. Look First
<font color=blue>**Question 3.1.**</font> Make a scatter plot with **`midparent`** on the horizontal axis and **`child`** on the vertical axis. No line yet.

In [ ]:
data.scatter('midparent', 'child')
plt.show()

<font color=green>Describe the **direction**, **form**, and **strength** of the association. Then guess the value of *r*. Does the cloud look more like the *r* = 0.3 or the *r* = 0.7 panel from the Lab 8 gallery?</font>

---

ANSWER

---

## 4. Fit the Line
<font color=blue>**Question 4.1.**</font> Use your toolbox to compute `my_r`, `my_slope`, and `my_intercept` for predicting **child** from **midparent**.

In [ ]:
my_r = correlation(data, 'midparent', 'child')
my_slope = slope(data, 'midparent', 'child')
my_intercept = intercept(data, 'midparent', 'child')
print(f'r = {my_r:.3f}')
print(f'child = {my_slope:.4f} × midparent + {my_intercept:.2f}')

<font color=green>Interpret your slope **in words, with units**: "For each additional ___ inch of midparent height, the child is predicted to change by ___ inches." How close was your guess for *r*?</font>

---

ANSWER

---

<font color=blue>**Question 4.2.**</font> A couple has a midparent height of **72 inches**. Predict their child's adult height. Store your prediction as `my_prediction`.

In [ ]:
my_prediction = my_slope * 72 + my_intercept
my_prediction

## 5. Check the Fit
A line can have a large *r* and still be the wrong model. Plot the **errors** (residuals), `actual y − predicted y`, against $x$. A good straight-line fit leaves errors that look like a patternless cloud centered on 0 ([Ch. 15.5](https://inferentialthinking.com/chapters/15/5/Visual_Diagnostics.html)).

<font color=blue>**Question 5.1.**</font> Compute the array `errors` and make a scatter plot of the errors against $x$, with a horizontal line at 0 (`plt.axhline(0)`).

<details><summary><b>💡 Hint</b></summary>

`errors = data.column(y) - (my_slope * data.column(x) + my_intercept)` — then `plt.scatter(...)` with the x column and `errors`. This is the `line_errors` idea from Lab 8, Question 2.3.

</details>

In [ ]:
x = data.column('midparent')
y = data.column('child')
errors = y - (my_slope * x + my_intercept)
plt.scatter(x, errors, s=12)
plt.axhline(0, color='red')
plt.xlabel('midparent'); plt.ylabel('error')
plt.show()

<font color=green>Do the errors look like a patternless cloud, or do you see a pattern (a curve, a fan, clusters, a few extreme points)? What does that tell you about your straight-line model?</font>

---

ANSWER

---

<font color=blue>**Question 5.2.**</font> Check your slope and intercept with `np.polyfit` (Lab 8, Part 6). Store `True` or `False` in `agrees`.

In [ ]:
coefficients = np.polyfit(data.column('midparent'), data.column('child'), 1)
agrees = np.isclose(coefficients.item(0), my_slope) and np.isclose(coefficients.item(1), my_intercept)
agrees

## 6. The Twist: Regression to the Mean, and a Hidden Variable
**Part A.** In standard units the regression line is $\hat{y}_{su} = r \cdot x_{su}$. Midparent height 72 inches is about 2.3 SDs above average.

<font color=blue>**Question 6.1.**</font> Compute `z_72` (72 inches in standard units of midparent height) and `z_child_pred` (the predicted child height in standard units). Is the child predicted to be more or less extreme than the parents?

In [ ]:
z_72 = (72 - np.mean(data.column('midparent'))) / np.std(data.column('midparent'))
z_child_pred = my_r * z_72
print(z_72, z_child_pred)

**Part B.** The `child` column mixes sons and daughters, and sons are taller on average.

<font color=blue>**Question 6.2.**</font> Make tables `sons` and `daughters`. Use your toolbox to compute `r_sons` and `r_daughters`, and the slope and intercept of each line.

In [ ]:
sons = data.where('gender', 'male')
daughters = data.where('gender', 'female')
r_sons = correlation(sons, 'midparent', 'child')
r_daughters = correlation(daughters, 'midparent', 'child')
print('sons:      r =', round(r_sons, 3), ' slope =', round(slope(sons, 'midparent', 'child'), 3))
print('daughters: r =', round(r_daughters, 3), ' slope =', round(slope(daughters, 'midparent', 'child'), 3))

<font color=green>1. Why is *r* larger for sons alone (or daughters alone) than for everyone together? Think about what the gender difference adds to the vertical spread.
2. Explain "regression to the mean" to a friend using your numbers from Question 6.1.
3. Galton called it regression toward *mediocrity*. Is anything mediocre about it?</font>

---

ANSWER

---

## 7. Your Final Figure
This is the product of your microproject. Your instructor will show the class gallery on screen, so make a figure a scientist would be proud of. **Requirements:**

- [ ] Scatter plot of the data
- [ ] Your regression line, labeled in the legend with its equation and *r*
- [ ] Axis labels **with units** (e.g. `midparent height (inches)` and `adult child height (inches)`)
- [ ] A title that states your **finding**, not just the variable names
- [ ] **Also** color sons and daughters differently and draw a separate regression line for each.

Build everything after `fig = plt.figure(...)` so `submit_plot` can save it.

<details><summary><b>💡 Starter pattern</b></summary>

```python
fig = plt.figure(figsize=(9, 5.5))
plt.scatter(x_values, y_values, s=14, alpha=0.6, label='data')
line_x = make_array(low, high)
plt.plot(line_x, my_slope * line_x + my_intercept, color='black', lw=2,
         label=f'y = {my_slope:.3f}x + {my_intercept:.1f},  r = {my_r:.2f}')
plt.xlabel('...'); plt.ylabel('...'); plt.title('...'); plt.legend()
plt.show()
```

</details>

In [ ]:
fig = plt.figure(figsize=(9, 5.5))
line_x = make_array(64, 76)
for group, color in [(sons, 'tab:blue'), (daughters, 'tab:orange')]:
    g = group.column('gender').item(0)
    s, b, rr = slope(group, 'midparent', 'child'), intercept(group, 'midparent', 'child'), correlation(group, 'midparent', 'child')
    plt.scatter(group.column('midparent'), group.column('child'), s=10, alpha=0.4, color=color)
    plt.plot(line_x, s * line_x + b, color=color, lw=2, label=f'{g}: y = {s:.2f}x {b:+.1f}, r = {rr:.2f}')
plt.plot(line_x, my_slope * line_x + my_intercept, color='black', ls='--', lw=2,
         label=f'everyone: y = {my_slope:.2f}x {my_intercept:+.1f}, r = {my_r:.2f}')
plt.xlabel('midparent height (inches)'); plt.ylabel('adult child height (inches)')
plt.title('Slopes below 1: children of tall parents regress toward the mean')
plt.legend(); plt.show()

<font color=blue>**Question 7.1.**</font> Write a **one-sentence caption** stating your main finding, then submit. You may submit again as often as you like — the newest version replaces the old one.

In [ ]:
caption = 'Splitting by gender raises r from 0.32 to about 0.5, and every slope is below 1: children of extreme parents are predicted to be closer to average.'
submit_plot(fig, name, 'MP2', r=my_r, slope=my_slope, intercept=my_intercept, caption=caption)

## 8. Reflection
<font color=green>In 3–5 sentences: What did the data say about your research question? How confident are you, and why? What was hardest to do **without** step-by-step instructions? Be ready to explain your figure to the class in 60 seconds.</font>

---

ANSWER

---

### 🏁 Done!
Save your notebook (💾). Your figure is already in the class gallery. Download and upload an HTML copy to Canvas if your instructor asks for one ([instructions](https://laserchemist.github.io/data/submission-instructions.html#html)).